# Part 1

In [1]:
import time, json, re
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

BASE = "http://books.toscrape.com/catalogue/"
HEADERS = {"User-Agent": "ADSP31011-session1-lab (student exercise; contact haok@uchicago.edu)"}
DATA = Path("data"); (DATA / "raw").mkdir(parents=True, exist_ok=True)

def get(url: str, pause: float = 1.0) -> requests.Response:
    """One polite GET: identify yourself, wait a second, raise on HTTP errors."""
    time.sleep(pause)
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = "utf-8"            # 4xx / 5xx become exceptions instead of silent bad data
    return r

print("pandas", pd.__version__)

pandas 2.3.1


In [2]:
r = get(BASE + "page-1.html")
print(r.status_code)
print(BeautifulSoup(r.text, "html.parser").select_one("p.price_color").get_text())


200
£51.77


In [3]:
from urllib.parse import urljoin
urljoin("http://books.toscrape.com/catalogue/page-1.html", "page-2.html")

soup = BeautifulSoup(r.text, "html.parser")
nxt = soup.select_one("li.next>a")
print(nxt)
print(nxt["href"])
print(urljoin(r.url, nxt["href"]))


<a href="page-2.html">next</a>
page-2.html
http://books.toscrape.com/catalogue/page-2.html


In [4]:
def parse_page(html: str) -> list[dict]:
    """All books on one catalogue page as a list of dicts (the same 5 keys as `book` above)."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for pod in soup.select("article.product_pod"):
        rows.append({
            "title":        pod.h3.a["title"],
            "price":        pod.select_one("p.price_color").get_text(strip=True),                      
            "rating":       pod.select_one("p.star-rating")["class"][1],                      
            "availability": pod.select_one("p.instock.availability").get_text(strip=True),
            "url":          BASE + pod.h3.a["href"],
        })
    return rows

def scrape_pages(start_url: str, max_pages: int = 5) -> pd.DataFrame:   # ← 改：参数从页码变成起始网址 + 页数上限
    all_rows = []
    url = start_url                                                      # ← 新增：从起始网址开始
    for n in range(1, max_pages + 1):                                    # ← 改：页码从 1 数到 max_pages
        try:
            html = get(url).text
        except requests.RequestException as e:                # a bad page must not kill the whole run
            msg = f"{datetime.now().isoformat(timespec='seconds')}  page {n}  {url}  {e}"
            print("LOGGED:", msg)                        # 屏幕上显示一份
            with open(DATA / "errors.log", "a", encoding="utf-8") as f:
                f.write(msg + "\n")                      # 文件里存一份
            break    
        (DATA / "raw" / f"page-{n}.html").write_text(html, encoding="utf-8")
        rows = parse_page(html)
        for row in rows:
            row["page"] = str(n)
        all_rows.extend(rows)
        print(f"page {n}: {len(rows)} books")

        nxt = BeautifulSoup(html, "html.parser").select_one("li.next>a")        # ← 新增 ① 找 next 链接
        if nxt is None:                                                          # ← 新增 ② 什么情况下停？
            break
        url = urljoin(url, nxt["href"])                                                       # ← 新增 ③ 下一页的完整网址
    return pd.DataFrame(all_rows, columns=["page", "title", "price", "rating", "availability", "url"])

books_raw = scrape_pages(BASE + "page-1.html", max_pages=5)              # ← 改：传起始网址
books_raw.head()

page 1: 20 books
page 2: 20 books
page 3: 20 books
page 4: 20 books
page 5: 20 books


,page,title,price,rating,availability,url
0,1,A Light in the Attic,£51.77,Three,In stock,http://books.toscrape.com/catalogue/a-light-in...
1,1,Tipping the Velvet,£53.74,One,In stock,http://books.toscrape.com/catalogue/tipping-th...
2,1,Soumission,£50.10,One,In stock,http://books.toscrape.com/catalogue/soumission...
3,1,Sharp Objects,£47.82,Four,In stock,http://books.toscrape.com/catalogue/sharp-obje...
4,1,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,http://books.toscrape.com/catalogue/sapiens-a-...


In [5]:
bad = scrape_pages(BASE + "page-999.html", max_pages=1)
bad

LOGGED: 2026-10-06T20:05:41  page 1  http://books.toscrape.com/catalogue/page-999.html  404 Client Error: Not Found for url: http://books.toscrape.com/catalogue/page-999.html


,page,title,price,rating,availability,url


In [6]:
# ✅ Checkpoint 1 — 100 rows, 6 columns, every column a string, every book once.
assert len(books_raw) == 100, f"expected 100 rows, got {len(books_raw)}"
assert set(books_raw.columns) == {"title", "price", "rating", "availability", "url", "page"}
assert (books_raw.dtypes == object).all(), "every column should still be a string"
assert books_raw["url"].is_unique, "same book appears twice"
books_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page          100 non-null    object
 1   title         100 non-null    object
 2   price         100 non-null    object
 3   rating        100 non-null    object
 4   availability  100 non-null    object
 5   url           100 non-null    object
dtypes: object(6)
memory usage: 4.8+ KB


Part 1 — Scrape 5 pages. The scraper follows the site's own "next" link instead of building page URLs by hand.

Bad pages: I chose break instead of continue. The URL of the next page comes from the "next" link inside the current page's HTML, so if a page fails to download, the scraper cannot know where the next page is. With continue, url would not be updated and the loop would just request the same bad page again. Instead, the error (time, page number, URL, reason) is written to data/errors.log, and the books already scraped are returned, so the run is not fatal.

# Part 2

In [7]:
OL = "https://openlibrary.org/search.json"
OL_HEADERS = {"User-Agent": "ADSP31011-session1-lab/1.0 (haok@uchicago.edu)"}   # Open Library asks callers to identify themselves

title = books_raw.loc[0, "title"]
resp = requests.get(OL, params={"title": title, "limit": 1,
                                "fields": "title,author_name,first_publish_year,subject,edition_count"},
                    headers=OL_HEADERS, timeout=20)
resp.raise_for_status()
payload = resp.json()
print("numFound:", payload["numFound"])
print(json.dumps(payload["docs"][0], indent=2))

numFound: 4
{
  "author_name": [
    "Shel Silverstein",
    "S. . Silverstein",
    "Xi Qiu Fu Si Tan (Shel Silverstein)"
  ],
  "edition_count": 27,
  "first_publish_year": 1981,
  "title": "A Light in the Attic",
  "subject": [
    "Humorous poetry",
    "American poetry",
    "Children's poetry, American",
    "Poetry, juvenile",
    "Poetry",
    "Juvenile literature",
    "Poetry (poetic works by one author)",
    "Spanish language materials",
    "Poes\u00eda infantil",
    "Poes\u00eda infantil estadounidense",
    "Poes\u00eda humor\u00edstica",
    "Humorous Stories",
    "JUVENILE FICTION",
    "Classics",
    "Humor",
    "Amerikanisches Englisch",
    "Kinderlyrik",
    "Po\u00e9sie humoristique",
    "Ouvrages pour la jeunesse",
    "Fiction",
    "Relations with women",
    "German Authors",
    "Women",
    "Translations into Hebrew",
    "Children's poetry"
  ]
}


In [8]:
doc = payload["docs"][0]
print("author:", (doc.get("author_name") or [None])[0])
print("n_subjects:", len(doc.get("subject") or []))

author: Shel Silverstein
n_subjects: 25


In [9]:
(DATA / "api").mkdir(exist_ok=True)                   # 存 JSON 的文件夹

def lookup_title(title: str, i: int) -> dict:
    """Best Open Library match for a title, flattened. Saves the raw JSON. No match -> a row of NaNs."""
    time.sleep(1.0)                                   # be polite — one request per second
    empty = {"title": title, "ol_title": None, "author": None,
             "first_publish_year": None, "edition_count": None, "n_subjects": None}
    params = {"title": title, "limit": 1,
              "fields": "title,author_name,first_publish_year,subject,edition_count"}
    try:
        resp = requests.get(OL, params=params, headers=OL_HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:            # a failed lookup must not kill the whole run
        msg = f"{datetime.now().isoformat(timespec='seconds')}  lookup {i}  {title}  {e}"
        print("LOGGED:", msg)
        with open(DATA / "errors.log", "a", encoding="utf-8") as f:
            f.write(msg + "\n")
        return empty
    (DATA / "api" / f"{i:02d}.json").write_text(resp.text, encoding="utf-8")   # save every response
    docs = resp.json()["docs"]
    if not docs:
        return empty                                  # no match -> our title + NaNs
    d = docs[0]                                       # the first (best) match
    return {
        "title":              title,                  # keep OUR title so we can join on it later
        "ol_title":           d.get("title"),
        "author":             (d.get("author_name") or [None])[0],
        "first_publish_year": d.get("first_publish_year"),
        "edition_count":      d.get("edition_count"),
        "n_subjects":         len(d.get("subject") or []),
    }


In [10]:
print(lookup_title(books_raw.loc[0, "title"], 1))

{'title': 'A Light in the Attic', 'ol_title': 'A Light in the Attic', 'author': 'Shel Silverstein', 'first_publish_year': 1981, 'edition_count': 27, 'n_subjects': 25}


In [11]:
lookups = []
for i, t in enumerate(books_raw["title"].head(20), start=1):
    row = lookup_title(t, i)
    print(("found  " if row["ol_title"] else "MISSING") + f"  {i:02d}  {t[:60]}")
    lookups.append(row)
ol = pd.DataFrame(lookups)
ol


found    01  A Light in the Attic
found    02  Tipping the Velvet
found    03  Soumission
found    04  Sharp Objects
found    05  Sapiens: A Brief History of Humankind
found    06  The Requiem Red
found    07  The Dirty Little Secrets of Getting Your Dream Job
MISSING  08  The Coming Woman: A Novel Based on the Life of the Infamous 
found    09  The Boys in the Boat: Nine Americans and Their Epic Quest fo
found    10  The Black Maria
MISSING  11  Starving Hearts (Triangular Trade Trilogy, #1)
found    12  Shakespeare's Sonnets
found    13  Set Me Free
MISSING  14  Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
found    15  Rip it Up and Start Again
MISSING  16  Our Band Could Be Your Life: Scenes from the American Indie 
found    17  Olio
MISSING  18  Mesaerion: The Best Science Fiction Stories 1800-1849
found    19  Libertarianism for Beginners
found    20  It's Only the Himalayas


,title,ol_title,author,first_publish_year,edition_count,n_subjects
0,A Light in the Attic,A Light in the Attic,Shel Silverstein,1981.0,27.0,25.0
1,Tipping the Velvet,Tipping the Velvet,Sarah Waters,1998.0,22.0,24.0
2,Soumission,Soumission,Michel Houellebecq,2015.0,21.0,10.0
3,Sharp Objects,Sharp Objects,Gillian Flynn,2006.0,31.0,26.0
4,Sapiens: A Brief History of Humankind,Summary of Sapiens : a Brief History of Humankind,BookNation,2020.0,1.0,0.0
5,The Requiem Red,The Requiem Red,Brynn Chapman,2016.0,2.0,18.0
6,The Dirty Little Secrets of Getting Your Dream...,The dirty little secrets of getting your dream...,Don Raskin,2016.0,2.0,4.0
7,The Coming Woman: A Novel Based on the Life of...,None,None,NaN,NaN,NaN
8,The Boys in the Boat: Nine Americans and Their...,Summary of the Boys in the Boat : Nine America...,BookNation,2020.0,1.0,0.0
9,The Black Maria,The black Maria,Aracelis Girmay,2016.0,1.0,16.0


In [12]:
# ✅ Checkpoint 2 — 20 rows (one per title, NaNs where no match), 20 saved JSON files, numeric year.
assert len(ol) == 20, f"expected 20 rows, got {len(ol)}"
assert ol["title"].tolist() == books_raw["title"].head(20).tolist(), "rows should match the first 20 books, in order"
n_json = len(list((DATA / "api").glob("*.json")))
assert n_json == 20, f"expected 20 JSON files, found {n_json}"
assert pd.api.types.is_numeric_dtype(ol["first_publish_year"]), "first_publish_year should be a number"
print(f"{ol['ol_title'].notna().sum()} of 20 titles matched")

15 of 20 titles matched


In [13]:
from urllib.parse import quote

check = ol[["title", "ol_title", "author", "first_publish_year"]].copy()
check["our_page"] = books_raw["url"].head(20)
check["ol_search"] = "https://openlibrary.org/search?q=" + check["title"].map(quote)
pd.set_option("display.max_colwidth", None)
check



,title,ol_title,author,first_publish_year,our_page,ol_search
0,A Light in the Attic,A Light in the Attic,Shel Silverstein,1981.0,http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,https://openlibrary.org/search?q=A%20Light%20in%20the%20Attic
1,Tipping the Velvet,Tipping the Velvet,Sarah Waters,1998.0,http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,https://openlibrary.org/search?q=Tipping%20the%20Velvet
2,Soumission,Soumission,Michel Houellebecq,2015.0,http://books.toscrape.com/catalogue/soumission_998/index.html,https://openlibrary.org/search?q=Soumission
3,Sharp Objects,Sharp Objects,Gillian Flynn,2006.0,http://books.toscrape.com/catalogue/sharp-objects_997/index.html,https://openlibrary.org/search?q=Sharp%20Objects
4,Sapiens: A Brief History of Humankind,Summary of Sapiens : a Brief History of Humankind,BookNation,2020.0,http://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,https://openlibrary.org/search?q=Sapiens%3A%20A%20Brief%20History%20of%20Humankind
5,The Requiem Red,The Requiem Red,Brynn Chapman,2016.0,http://books.toscrape.com/catalogue/the-requiem-red_995/index.html,https://openlibrary.org/search?q=The%20Requiem%20Red
6,The Dirty Little Secrets of Getting Your Dream Job,The dirty little secrets of getting your dream job,Don Raskin,2016.0,http://books.toscrape.com/catalogue/the-dirty-little-secrets-of-getting-your-dream-job_994/index.html,https://openlibrary.org/search?q=The%20Dirty%20Little%20Secrets%20of%20Getting%20Your%20Dream%20Job
7,"The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull",None,None,NaN,http://books.toscrape.com/catalogue/the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.html,https://openlibrary.org/search?q=The%20Coming%20Woman%3A%20A%20Novel%20Based%20on%20the%20Life%20of%20the%20Infamous%20Feminist%2C%20Victoria%20Woodhull
8,The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics,Summary of the Boys in the Boat : Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics,BookNation,2020.0,http://books.toscrape.com/catalogue/the-boys-in-the-boat-nine-americans-and-their-epic-quest-for-gold-at-the-1936-berlin-olympics_992/index.html,https://openlibrary.org/search?q=The%20Boys%20in%20the%20Boat%3A%20Nine%20Americans%20and%20Their%20Epic%20Quest%20for%20Gold%20at%20the%201936%20Berlin%20Olympics
9,The Black Maria,The black Maria,Aracelis Girmay,2016.0,http://books.toscrape.com/catalogue/the-black-maria_991/index.html,https://openlibrary.org/search?q=The%20Black%20Maria


## Hand-check
I checked five matches by opening both links and comparing the title, author and year. Two are correct: *A Light in the Attic* and *Sharp Objects*. Three are wrong. For *Sapiens* and *The Boys in the Boat*, Open Library gave me summary books by BookNation, not the real books. For *Shakespeare's Sonnets*, the title is the same, but the book is a collection of essays about the sonnets by James Schiffer. This shows that the same title does not always mean the same book. Searching with the title and the author, or with the ISBN, would give better matches.



# Part 3

In [14]:
books_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page          100 non-null    object
 1   title         100 non-null    object
 2   price         100 non-null    object
 3   rating        100 non-null    object
 4   availability  100 non-null    object
 5   url           100 non-null    object
dtypes: object(6)
memory usage: 4.8+ KB


In [15]:
RATING = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

books = books_raw.copy()
books["price"]      = books["price"].str.replace("£", "", regex=False).astype(float)
books["rating"]     = books["rating"].map(RATING).astype("Int64")
books["in_stock"]   = books["availability"].str.contains("In stock", case=False)
books["page"]       = books["page"].astype(int)
books["scraped_at"] = pd.Timestamp.now(tz="UTC")
books.head()


,page,title,price,rating,availability,url,in_stock,scraped_at
0,1,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,True,2026-10-07 01:06:15.427756+00:00
1,1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,True,2026-10-07 01:06:15.427756+00:00
2,1,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission_998/index.html,True,2026-10-07 01:06:15.427756+00:00
3,1,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-objects_997/index.html,True,2026-10-07 01:06:15.427756+00:00
4,1,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,True,2026-10-07 01:06:15.427756+00:00


In [16]:
# de-duplicate: url is each book's identity (two different books can share a title)
n_before = len(books)
books = books.drop_duplicates(subset="url").reset_index(drop=True)
print(f"duplicates removed: {n_before - len(books)}")

# LEFT join: every scraped book stays; API fields are NaN where there was no lookup or no match
merged = books.merge(ol, on="title", how="left")
merged.info()      # after — compare with books_raw.info() above

duplicates removed: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   page                100 non-null    int64              
 1   title               100 non-null    object             
 2   price               100 non-null    float64            
 3   rating              100 non-null    Int64              
 4   availability        100 non-null    object             
 5   url                 100 non-null    object             
 6   in_stock            100 non-null    bool               
 7   scraped_at          100 non-null    datetime64[us, UTC]
 8   ol_title            15 non-null     object             
 9   author              15 non-null     object             
 10  first_publish_year  15 non-null     float64            
 11  edition_count       15 non-null     float64            
 12  n_subjects     

In [17]:
by_rating = (merged.groupby("rating")
                   .agg(n_books=("title", "count"), avg_price=("price", "mean"))
                   .round(2))
by_rating

,n_books,avg_price
rating,,
1,22,35.52
2,19,35.91
3,22,36.84
4,18,33.98
5,19,30.01


In [18]:
merged["first_publish_decade"] = (merged["first_publish_year"] // 10 * 10).astype("Int64")

by_decade = (merged.groupby("first_publish_decade")
                   .agg(n_books=("title", "count"), avg_price=("price", "mean"))
                   .round(2))
by_decade


,n_books,avg_price
first_publish_decade,,
1980,1,51.77
1990,3,30.62
2000,2,41.42
2010,7,39.80
2020,2,38.42


In [19]:
merged.to_csv(DATA / "books_silver.csv", index=False)
merged.to_parquet(DATA / "books_silver.parquet", index=False)

In [20]:
# ✅ Checkpoint 3 — 100 unique books, dtypes changed, no £ left, both files written.
assert len(merged) == 100 and merged["url"].is_unique
assert merged["price"].dtype == "float64" and not merged["price"].astype(str).str.contains("£").any()
assert str(merged["rating"].dtype) == "Int64"
assert merged["in_stock"].dtype == bool
assert (DATA / "books_silver.csv").exists() and (DATA / "books_silver.parquet").exists()
print("missing first_publish_year:", merged["first_publish_year"].isna().sum(), "of", len(merged))
merged.info()


missing first_publish_year: 85 of 100
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   page                  100 non-null    int64              
 1   title                 100 non-null    object             
 2   price                 100 non-null    float64            
 3   rating                100 non-null    Int64              
 4   availability          100 non-null    object             
 5   url                   100 non-null    object             
 6   in_stock              100 non-null    bool               
 7   scraped_at            100 non-null    datetime64[us, UTC]
 8   ol_title              15 non-null     object             
 9   author                15 non-null     object             
 10  first_publish_year    15 non-null     float64            
 11  edition_count         15 non-null 

In [21]:
merged

,page,title,price,rating,availability,url,in_stock,scraped_at,ol_title,author,first_publish_year,edition_count,n_subjects,first_publish_decade
0,1,A Light in the Attic,51.77,3,In stock,http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html,True,2026-10-07 01:06:15.427756+00:00,A Light in the Attic,Shel Silverstein,1981.0,27.0,25.0,1980
1,1,Tipping the Velvet,53.74,1,In stock,http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html,True,2026-10-07 01:06:15.427756+00:00,Tipping the Velvet,Sarah Waters,1998.0,22.0,24.0,1990
2,1,Soumission,50.10,1,In stock,http://books.toscrape.com/catalogue/soumission_998/index.html,True,2026-10-07 01:06:15.427756+00:00,Soumission,Michel Houellebecq,2015.0,21.0,10.0,2010
3,1,Sharp Objects,47.82,4,In stock,http://books.toscrape.com/catalogue/sharp-objects_997/index.html,True,2026-10-07 01:06:15.427756+00:00,Sharp Objects,Gillian Flynn,2006.0,31.0,26.0,2000
4,1,Sapiens: A Brief History of Humankind,54.23,5,In stock,http://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html,True,2026-10-07 01:06:15.427756+00:00,Summary of Sapiens : a Brief History of Humankind,BookNation,2020.0,1.0,0.0,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,5,Lumberjanes Vol. 3: A Terrible Plan (Lumberjanes #9-12),19.92,2,In stock,http://books.toscrape.com/catalogue/lumberjanes-vol-3-a-terrible-plan-lumberjanes-9-12_905/index.html,True,2026-10-07 01:06:15.427756+00:00,NaN,NaN,NaN,NaN,NaN,<NA>
96,5,"Layered: Baking, Building, and Styling Spectacular Cakes",40.11,1,In stock,http://books.toscrape.com/catalogue/layered-baking-building-and-styling-spectacular-cakes_904/index.html,True,2026-10-07 01:06:15.427756+00:00,NaN,NaN,NaN,NaN,NaN,<NA>
97,5,Judo: Seven Steps to Black Belt (an Introductory Guide for Beginners),53.90,2,In stock,http://books.toscrape.com/catalogue/judo-seven-steps-to-black-belt-an-introductory-guide-for-beginners_903/index.html,True,2026-10-07 01:06:15.427756+00:00,NaN,NaN,NaN,NaN,NaN,<NA>
98,5,Join,35.67,5,In stock,http://books.toscrape.com/catalogue/join_902/index.html,True,2026-10-07 01:06:15.427756+00:00,NaN,NaN,NaN,NaN,NaN,<NA>
